[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/srini2k03/jobpost-extractor-qlora/blob/main/notebooks/04_eval_predict.ipynb)

# 04 · Fine-tuned model: predictions + merge

Loads the base model in 16-bit with the pushed LoRA adapter, writes predictions on every test split (free decoding on the full split, guided decoding on the shared subset), then merges the adapter and pushes the merged model that the GGUF/AWQ steps quantise. Same code as `make local-eval` (`jobx.pipeline.eval_finetuned`).

**How to run**
1. Runtime → Change runtime type → **T4 GPU**.
2. Secrets (key icon in the left sidebar), each with *Notebook access* switched on: `HF_TOKEN`, `HF_USERNAME`.
   `HF_USERNAME` is your Hugging Face account name; `HF_TOKEN` needs *write* access.
3. Runtime → **Run all**. Expected runtime: **~45–60 min**. Cells after the setup are safe to re-run.

**Outputs:** predictions + metrics for `qwen2.5-1.5b_qlora` and `qwen2.5-1.5b_qlora_guided` pushed to `<HF_USERNAME>/jobx-artifacts`; merged model pushed to `<HF_USERNAME>/jobx-qwen2.5-1.5b-merged`.

All code lives in `src/jobx` of the GitHub repo; this notebook only orchestrates it.

In [ ]:
# Setup: clone the repo (code + data), install pinned dependencies, install jobx itself.
import os, sys
IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/srini2k03/jobpost-extractor-qlora.git"
if IN_COLAB:
    if not os.path.exists("/content/jobpost-extractor-qlora"):
        !git clone --depth 1 {REPO_URL} /content/jobpost-extractor-qlora
    %cd /content/jobpost-extractor-qlora
    !pip install -q transformers==4.56.2 peft==0.17.1 trl==0.23.1 bitsandbytes==0.48.1 accelerate==1.10.1 datasets==4.1.1 huggingface_hub==0.35.3 wandb==0.22.2 lm-format-enforcer==0.11.3
    !pip install -q -e .
else:
    # running locally from a checkout: notebooks/ -> repo root
    if os.path.basename(os.getcwd()) == "notebooks":
        os.chdir("..")
import torch
assert torch.cuda.is_available(), 'No GPU: Runtime -> Change runtime type -> T4 GPU'
print(torch.__version__, torch.cuda.get_device_name(0))


In [ ]:
# Secrets -> environment variables (read by jobx.hub and wandb).
import os
for key in ("HF_TOKEN", "HF_USERNAME", "WANDB_API_KEY"):
    if not os.environ.get(key):
        try:
            from google.colab import userdata
            os.environ[key] = userdata.get(key)
        except Exception:
            print(f"secret {key} not available")
from jobx.hub import hf_username, repo_id
if os.environ.get("HF_TOKEN"):
    from huggingface_hub import login
    login(token=os.environ["HF_TOKEN"], add_to_git_credential=False)
print("Hub account:", hf_username(), "| artifacts repo:", repo_id("artifacts"))

In [ ]:
RUN_LLAMA = False
from jobx.hub import repo_id
from jobx.pipeline import eval_finetuned, load_profile
PROFILE = load_profile('colab_t4')
ADAPTER = repo_id('adapter')
ADAPTER

## Predictions (free + guided), merge, push

In [ ]:
merged = eval_finetuned(PROFILE, adapter=ADAPTER, push=True)
merged

## Optional: Llama-3.2-1B fine-tune predictions

In [ ]:
if RUN_LLAMA:
    from jobx.baselines.hf_runner import load_model
    from jobx.baselines.runs import available_test_splits, run_approach
    lmodel, ltok = load_model('meta-llama/Llama-3.2-1B-Instruct', adapter=repo_id('adapter_llama'), merge_adapter=True)
    for split in available_test_splits():
        run_approach(lmodel, ltok, 'llama3.2-1b_qlora', split, model_id=repo_id('adapter_llama'), batch_size=16)

## Next
GitHub → Actions → **quantize-benchmark** → Run workflow (GGUF Q8_0/Q4_K_M on CPU), and `05_awq_benchmark` for the GPU rows.